[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/purva-thakre/qce26-qrisp/blob/Pietro_notebooks_Colab_links/Session_2_Introduction_to_Jasp/JaspTracing.ipynb)

# JAX Internals and The Execution Model

## Setup

In [1]:
!pip install qrisp

Let's import JAX (installed by default with qrisp) and check that it works: 

In [ ]:
import jax
import jax.numpy as jnp
import time

Notice that `jaxlib` is always supposed to be installed with `jax` as it contains the compiled XLA (the actual compiler that turns a jaxpr into machine code for a specific device) along with many other objects required by JAX for just-in-time compilation. 

Version `0.7.1` is the JAX version currently pinned within qrisp.

In [41]:
print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

JAX version: 0.7.1
Available devices: [CpuDevice(id=0)]


## Our first Jaxpr

The `jaxpr` is the (first) intermediate representation that JAX uses to just-in-time compile functions.

We choose a very simple function `f` and we *trace it* to obtain the JAX intermediate representation for it:

In [43]:
def f(x):
    return jnp.sin(x) + x * 3.0

`jax.make_jaxpr` reveals the intermediate representation JAX builds for a function, before any real numbers are involved.

In [50]:
jaxpr = jax.make_jaxpr(f)(jnp.array(20.0))
print(jaxpr)

{ lambda ; a:f64[]. let
    b:f64[] = sin a
    c:f64[] = mul a 3.0:f64[]
    d:f64[] = add b c
  in (d,) }


Observations:

- The `jaxpr` only stores information about the **type** and **shape** of the *traced* variables (`a`, `b`, `c`, `d` above), not their concrete numbers. The `3.0` we see printed with its actual value isn't a traced variable at all: it's a plain Python literal.

- Not all the instructions of our functions can be traced this way. Try, for example, inserting a `print` statement in the function above. It won't raise an error, but its output won't show up anywhere in the printed jaxpr either (we'll see exactly why in the "Side Effects" section below).

- Everything that is not recorded in the `jaxpr` in the tracing stage is **completely ignored** by JAX in the compilation pipeline

## Anatomy of a Jaxpr

This is the general structure of a Jaxpr

```
jaxpr ::= { lambda <constvars> ; <invars>. 
            let <equations>
            in  <expressions> }
```

We have to make a distinction here:

- `jax.make_jaxpr` actually returns a `ClosedJaxpr`. 

- The `jaxpr` attribute unwraps it to the inner `Jaxpr` holding the real `let` equations.
When we say `jaxpr` we typically refer to the inner `Jaxpr`.

Printing either one gives the same text:

In [6]:
print(f"jaxpr: {jaxpr}")
print(f"type of jaxpr: {type(jaxpr).__name__}\n\n\n")
print(f"jaxpr.jaxpr: {jaxpr.jaxpr}")
print(f"type of jaxpr.jaxpr: {type(jaxpr.jaxpr).__name__}")

jaxpr: { lambda ; a:f32[]. let
    b:f32[] = sin a
    c:f32[] = mul a 3.0:f32[]
    d:f32[] = add b c
  in (d,) }
type of jaxpr: ClosedJaxpr



jaxpr.jaxpr: { lambda ; a:f32[]. let
    b:f32[] = sin a
    c:f32[] = mul a 3.0:f32[]
    d:f32[] = add b c
  in (d,) }
type of jaxpr.jaxpr: Jaxpr


A `Jaxpr` is nothing but a Python class, and we can access several attributes, including `invars`, `outvars`, `eqns`, etc.

In [51]:
jaxpr

{ lambda ; a:f64[]. let
    b:f64[] = sin a
    c:f64[] = mul a 3.0:f64[]
    d:f64[] = add b c
  in (d,) }

In [52]:
print(f"Input variables (invars): {jaxpr.jaxpr.invars}")
print(f"Output variables (outvars): {jaxpr.jaxpr.outvars}")
print(f"Equations: {jaxpr.jaxpr.eqns}")

Input variables (invars): [Var(id=134831272051776):float64[]]
Output variables (outvars): [Var(id=134831250344448):float64[]]
Equations: [a:f64[] = sin b, a:f64[] = mul b 3.0:f64[], a:f64[] = add b c]


`jaxpr.jaxpr.eqns` returns a list of equations, and we can iterate over them:

In [9]:
for eqn in jaxpr.jaxpr.eqns:
    print(
        f" primitive: {eqn.primitive.name}",
        f"| inputs: {eqn.invars}",
        f"| outvars: {eqn.outvars}",
    )

 primitive: sin | inputs: [Var(id=134835883468288):float32[]] | outvars: [Var(id=134835806885632):float32[]]
 primitive: mul | inputs: [Var(id=134835883468288):float32[], Literal(3.0)] | outvars: [Var(id=134835806888896):float32[]]
 primitive: add | inputs: [Var(id=134835806885632):float32[], Var(id=134835806888896):float32[]] | outvars: [Var(id=134835806891904):float32[]]


Each equation has several attributes, including **primitive**, plus its input/output variables.

In qrisp we have the *exact same* `jaxpr` representation, called `jaspr`. Internally, the `Jaspr` is just a Python class that inherits from the native `ClosedJaxpr` class from JAX.

Similarly to a `ClosedJaxpr`, a `jaspr` can be created using the `make_jaspr` function:

In [53]:
from qrisp import QuantumVariable, make_jaspr, x, cx, measure


def circuit(n_qubits):
    qv = QuantumVariable(n_qubits)
    x(qv[0])
    cx(qv[0], qv[1])
    meas_res = measure(qv)
    return meas_res


jaspr = make_jaspr(circuit)(4)
print(jaspr)

{ lambda ; a:i64[] b:QuantumState. let
    c:QubitArray d:QuantumState = jasp.create_qubits a b
    e:Qubit = jasp.get_qubit c 0:i64[]
    f:QuantumState = jasp.quantum_gate[gate=x] e d
    g:Qubit = jasp.get_qubit c 1:i64[]
    h:QuantumState = jasp.quantum_gate[gate=cx] e g f
    i:i64[] j:QuantumState = jasp.measure c h
  in (i, j) }


There are a few differences compared to a native `jaxpr`:

- In a `jaspr` we also have **quantum primitives** (that is, we define new primitives and these appear in the jax expression): `create_qubits`, `get_qubit`, `quantum_gate`, and `measure` above are all examples. `QuantumState`, `Qubit`, and `QubitArray` are the new *types* these primitives introduce, the quantum equivalent of `f32[]`.

- The last (hidden) invar of the `jaspr` and the last (hidden) outvar is always a `QuantumState`, that is threaded along the equations to simulate our quantum circuit. This follows from the same rule we already saw: a jaxpr only ever sees what flows through explicit variables, so a gate cannot silently mutate some hidden quantum state, the same way a `print` statement cannot silently leave a trace. The state has to be passed in and returned explicitly, just like any other value.

## Constants (bonus)

Not everything a traced function uses comes from its arguments. Closed-over values get captured too. 

Let's see how this shows up in the jaxpr:

In [11]:
# This `matrix` is closed over by f, not passed as an argument
matrix = jnp.ones((3, 3))


def f(x):
    return matrix @ x

In [12]:
jaxpr = jax.make_jaxpr(f)(jnp.ones(3))
print(f"jaxpr: {jaxpr}\n")
print(f"consts: {jaxpr.consts}")

jaxpr: { lambda a:f64[3,3]; b:f64[3]. let
    c:f64[3] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float64
    ] a b
  in (c,) }

consts: [Array([[1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.]], dtype=float64)]


There it is: `a:f32[3,3]` is `matrix`, listed separately from the real argument `b`.

Its concrete value lives in `jaxpr.consts` rather than being passed in on every call.

Putting it together, a jaxpr only ever has three kinds of things sitting where a number could be:

- a **traced variable**: a placeholder with no value until runtime (`a`, `b`, `c`, `d` in our very first example),

- a **literal**: a bare number written directly in your source (the `3.0` in that same example), baked into the equation as-is,

- a **constant**: a closed-over value like `matrix` here, listed separately as its own variable with the real value sitting in `.consts`.

Only the traced variables are genuinely abstract. 

Literals and constants both carry their real value along in the printed jaxpr, just via two different mechanisms.

## General Structure of a Primitive (bonus)

A primitive is JAX's smallest unit of computation (`sin`, `mul`, `add`, `dot_general`, and so on), each with a fixed input/output signature and nothing further to trace into.

This is the general form of a primitive:

```
equation ::= let <output vars> = <primitive> [<params>] <input exprs>
```

The optional `[params]` are extra static configuration for that operation, like the `dimension_numbers` attached to `dot_general` earlier.

Depending on the *context* (tracing, eager execution, lowering, vectorization, differentiation, etc.) JAX knows how to treat a certain primitive based on specific *rules*.

In the `Primitives` notebook (in the extra section) we see how we can define new primitives (and rules associated with them). This is exactly what the Jasp module in qrisp does to define quantum primitives (introducing new types, such as `QuantumState`) within the jax expressions.

## Evaluating a Jaxpr (bonus)

We can evaluate a `jaxpr` with `jax.core.eval_jaxpr`, given the constants and the real arguments by hand:

In [13]:
from jax.core import eval_jaxpr

result = eval_jaxpr(jaxpr.jaxpr, jaxpr.consts, jnp.ones(3))
print(f"eval_jaxpr result: {result}")
print(f"Direct call result: {f(jnp.ones(3))}")

eval_jaxpr result: [Array([3., 3., 3.], dtype=float64)]
Direct call result: [3. 3. 3.]


The difference between the two:

- Calling `f(...)` directly re-executes the *original Python function* every time: any Python-level behavior in its body (prints, side effects, control flow) runs again, exactly as it did during tracing.

- `eval_jaxpr(...)` instead interprets the already-recorded jaxpr equations directly. It never touches the original Python source again: it only knows about the primitive ops that got captured during tracing.

For a **pure** function like `f` here, both give the same numerical result, as the two prints above confirm.

The difference only becomes observable once a function does something a jaxpr can't capture (we will see examples below).

In qrisp we are always evaluating or *interpreting* jaxprs containing both classical and quantum primitives (that is, primitives defined within the Jasp module).

## Tracing with Abstract Values

When JAX traces a function it doesn't run it on real data.

Each argument becomes a placeholder that only carries shape and dtype, as we mentioned before.

Anything a Python `print` shows during tracing reflects that placeholder, not the concrete values you called with.

In [54]:
def spy_on_tracing(x):
    print(f"Type of x during tracing: {type(x).__name__}")
    print(f"Value of x during tracing: {x}")
    return 2.0 * x

In [55]:
jax.make_jaxpr(spy_on_tracing)(jnp.array([6.0, 12.0]))

Type of x during tracing: DynamicJaxprTracer
Value of x during tracing: JitTracer<float64[2]>


{ lambda ; a:f64[2]. let b:f64[2] = mul 2.0:f64[] a in (b,) }

Observations:

- Notice the printed type is a tracer (`DynamicJaxprTracer`), and the value shown is a placeholder (not the concrete object we provided). `JitTracer<float64[2]>` is just how that same tracer prints itself, not a different object.

- Notice that in the `jaxpr` there is *no trace* of our print statements. Therefore, JAX will completely ignore its existence in the lowering process.

## Python Control Flow

### Python Loops

Python `for` loops leave no trace of themselves in a jaxpr, just like any other Python control flow. JAX just runs the loop body at trace time and records whatever ops happen, once per iteration.


With a static bound, that's indistinguishable from writing the same line three times by hand:

In [58]:
def loop_function(x):
    for _ in range(3):
        x = jnp.tanh(x)
    return x


jaxpr = jax.make_jaxpr(loop_function)(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f64[]. let
    b:f64[] = tanh a
    c:f64[] = tanh b
    d:f64[] = tanh c
  in (d,) }


In [17]:
# same as loop_function above, just written out by hand for comparison
def loop_function_alternative(x):
    x = jnp.tanh(x)
    x = jnp.tanh(x)
    x = jnp.tanh(x)
    return x


jaxpr = jax.make_jaxpr(loop_function_alternative)(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f64[]. let
    b:f64[] = tanh a
    c:f64[] = tanh b
    d:f64[] = tanh c
  in (d,) }


We have the same jaxpr both ways: three chained `tanh` equations, no looping construct in sight.

Therefore, JAX cannot 'see' the difference: it cannot detect the presence of the loop in the first place.

Furthermore, what if the bound of the loop is a tracer?

In [59]:
def loop_function_broken(y, x):
    for _ in range(y):
        x = jnp.tanh(x)
    return x


try:
    jax.make_jaxpr(loop_function_broken)(3, jnp.array(1.0))
except Exception as e:
    print(type(e).__name__)
    print(e)

TracerIntegerConversionError
The __index__() method was called on traced array with shape int64[]
The error occurred while tracing the function loop_function_broken at /tmp/ipykernel_35700/2190278666.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument y.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerIntegerConversionError


`range(y)` needs a concrete Python integer to know how many times to loop, but `y` is a tracer, a placeholder with no concrete value until runtime.

The fix comes from the `jax.lax` module. We can use `jax.lax.fori_loop`:

In [60]:
def dynamic_loop(x, n):
    def body(_, val):  # first arg is the loop index, unused here
        return jnp.tanh(val)

    return jax.lax.fori_loop(0, n, body, x)

In [62]:
jaxpr = jax.make_jaxpr(dynamic_loop)(jnp.array(1.0), 3)
print(jaxpr)

{ lambda ; a:f64[] b:i64[]. let
    _:i64[] _:i64[] c:f64[] = while[
      body_jaxpr={ lambda ; d:i64[] e:i64[] f:f64[]. let
          g:i64[] = add d 1:i64[]
          h:f64[] = tanh f
        in (g, e, h) }
      body_nconsts=0
      cond_jaxpr={ lambda ; i:i64[] j:i64[] k:f64[]. let
          l:bool[] = lt i j
        in (l,) }
      cond_nconsts=0
    ] 0:i64[] b a
  in (c,) }


Observations:

- The loop is now recorded in the `jaxpr` since we have the `while` primitive in the body. We have `while` because `jax.lax.fori_loop` is a convenience wrapper that dispatches to the `while` primitive (or to `scan`, when the loop bound is static).

- The jaxpr stays small since we have a single `while` primitive. That is, even with `n=10000`, nothing gets unrolled this time.

- Therefore, the **size of the intermediate representation and the compilation time do not depend on the size of the loop!**

### Conditional Statements (bonus)

The fact that during tracing we cannot keep track of the concrete values associated with our arguments has consequences for conditional statements as well (same as in the loop case).

For example, what if we have a Python `if` statement in the body of our function in which the predicate is a tracer?

Plain Python `if` needs a concrete boolean right away, but during tracing, `x > 0` is itself a traced value, not a real bool yet.


Here's what happens when a traced function tries to branch on that traced condition:

In [21]:
def broken_function(x):
    if x > 0:
        return x * 2.0
    else:
        return x

In [22]:
try:
    jax.make_jaxpr(broken_function)(jnp.array(1.0))
except Exception as e:
    print(type(e).__name__)
    print(e)

TracerBoolConversionError
Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function broken_function at /tmp/ipykernel_35700/1210078963.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError


The issue is that JAX can't convert a traced value to a Python `bool` to decide the `if`... since no concrete answer exists yet at trace time.

We can fix this with a primitive that JAX provides in the `jax.lax` module. Specifically, we can use the `jax.lax.cond` higher order primitive.

A higher-order primitive is a primitive that receives a function as input.

The fix below swaps that Python `if` for `jax.lax.cond`, which encodes *both* branches into the jaxpr and picks one at runtime instead.

In [23]:
def fixed_function(x):
    return jax.lax.cond(x > 0, lambda x: x * 2.0, lambda x: x, x)

In [24]:
jaxpr = jax.make_jaxpr(fixed_function)(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f64[]. let
    b:bool[] = gt a 0.0:f64[]
    c:i32[] = convert_element_type[new_dtype=int32 weak_type=False] b
    d:f64[] = cond[
      branches=(
        { lambda ; e:f64[]. let  in (e,) }
        { lambda ; f:f64[]. let g:f64[] = mul f 2.0:f64[] in (g,) }
      )
    ] c a
  in (d,) }


Observations:

- Both branches now show up inside the `cond` primitive's `branches=(...)` within the `jaxpr`.

- Which branch to take depends on which value we provide at *runtime*: the decision moved from trace time (or compilation time) to run time.

- Even if we had provided a concrete predicate to our Python `if` control flow, the control flow itself is not recorded in the `jaxpr`. Therefore, JAX would have never known that a conditional branch was present in the first place. We already saw the same behaviour concretely above with native `for` loops.

## Static vs. Dynamic Values

So far every argument became a traced placeholder. 

Sometimes we want an argument treated as a fixed Python value baked directly into the jaxpr instead. That's what `static_argnums` is for.

Let's define a function with 2 arguments:

In [63]:
def uses_n(x, n):
    return x * n

By default, both arguments are dynamic (that is, they become tracers):

In [64]:
jaxpr_dynamic = jax.make_jaxpr(uses_n)(jnp.array(1.0), 2.0)
print(jaxpr_dynamic)

{ lambda ; a:f64[] b:f64[]. let c:f64[] = mul a b in (c,) }


Let's declare the second one static:

In [66]:
jaxpr_static = jax.make_jaxpr(uses_n, static_argnums=1)(jnp.array(1.0), 2.0)
print(jaxpr_static)

{ lambda ; a:f64[]. let b:f64[] = mul a 2.0:f64[] in (b,) }


Now `2.0` is baked in as a literal and there's no second input at all.

Crucially, a different jaxpr gets built for each value of `n`. In other words, JAX needs to re-trace from scratch.

Therefore, we **cannot just-in-time compile** a function if we provide a different value of a static argument.

## Side Effects at Trace Time vs. Runtime (bonus)

Tracing runs our Python function's code once to build the `jaxpr`, so any side effect in there (printing, mutating a list) still fires during that trace, even though no 'real' computation happened yet.

This might lead to confusing behaviour.

Watch the counter to see exactly when the side effect fires (once for tracing, once for a normal call).

In [28]:
counter = [0]


def side_effect_function(x):
    counter[0] += 1
    print(f"counter[0]: {counter[0]}")
    return x * 2.0

In [29]:
print("=== Tracing ===")
jaxpr = jax.make_jaxpr(side_effect_function)(jnp.array(1.0))
print(jaxpr)
print()
print(f"Counter value: {counter[0]}")

=== Tracing ===
counter[0]: 1
{ lambda ; a:f64[]. let b:f64[] = mul a 2.0:f64[] in (b,) }

Counter value: 1


In [30]:
print("=== Normal call ===")
_ = side_effect_function(jnp.array(1.0))
print(f"Counter value: {counter[0]}")

=== Normal call ===
counter[0]: 2
Counter value: 2


The side effect fired once while `make_jaxpr` traced the function (counter -> 1), then again on the direct call (counter -> 2). 

Tracing is a real execution of our code, not a dry run.

As a rule of thumb, we should always rely on the `jaxpr` structure to know what JAX can 'see'.

## Pytrees: Structured Inputs and Outputs (bonus)

Real functions rarely take one flat array. They take nested structures like this dict of weights and biases. JAX calls any such nested container of arrays a *pytree*.


`jax.make_jaxpr` handles it transparently: under the hood it just flattens the pytree into individual traced leaves:

In [31]:
params = {"w": jnp.ones((3, 2)), "b": jnp.zeros((3,))}


def linear(params, x):
    return params["w"] @ x + params["b"]

In [32]:
jaxpr = jax.make_jaxpr(linear)(params, jnp.ones((2)))
print(jaxpr)

{ lambda ; a:f64[3] b:f64[3,2] c:f64[2]. let
    d:f64[3] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float64
    ] b c
    e:f64[3] = add d a
  in (e,) }


The jaxpr only has flat scalar/array binders (`a`, `b`, `c`): the dict structure of `params` is gone once you're looking at the jaxpr, flattened down to its leaves.


Here's the same computation again, written with `w` and `b` as separate plain arguments instead of a dict, just to confirm it's really only the leaves that matter:

In [33]:
def linear_alternative(w, b, x):
    return w @ x + b

In [34]:
jaxpr = jax.make_jaxpr(linear_alternative)(params["w"], params["b"], jnp.ones((2)))
print(jaxpr)

{ lambda ; a:f64[3,2] b:f64[3] c:f64[2]. let
    d:f64[3] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float64
    ] a c
    e:f64[3] = add d b
  in (e,) }


Structurally identical to the jaxpr above (one `dot_general`, one `add`): the dict was a Python-level convenience only, not something that changes the underlying computation.

Those flatten/unflatten steps JAX does for you automatically are also available directly, through `jax.tree_util`.

In [35]:
leaves, treedef = jax.tree_util.tree_flatten(params)
print(f"Leaves (shapes): {[l.shape for l in leaves]}")
print(f"Treedef: {treedef}")

Leaves (shapes): [(3,), (3, 2)]
Treedef: PyTreeDef({'b': *, 'w': *})


`tree_flatten` gives you the leaves in a canonical order plus a `treedef` describing how to rebuild the original structure, dict keys and all.

In [36]:
params_back = treedef.unflatten(leaves)
print(f"params_back: {params_back}")

params_back: {'b': Array([0., 0., 0.], dtype=float64), 'w': Array([[1., 1.],
       [1., 1.],
       [1., 1.]], dtype=float64)}


In [37]:
scaled_params = jax.tree_util.tree_map(lambda p: p * 0.01, params)
print(f"scaled_params: {scaled_params}")

scaled_params: {'b': Array([0., 0., 0.], dtype=float64), 'w': Array([[0.01, 0.01],
       [0.01, 0.01],
       [0.01, 0.01]], dtype=float64)}


`tree_map` is really just flatten -> apply the function to every leaf -> unflatten, the same steps you just did by hand, wrapped into one call.

## Resource Estimation at Scale

Jasp also ships decorators that compile a function into a *resource estimator* instead of a simulator or a real quantum computer.

For example, `count_ops` traces the function once and returns a dictionary of gate counts, without ever running an actual quantum simulation.

This works for exactly the reason we just saw with static vs. dynamic values: if the size of a circuit is a *dynamic* argument (a traced value, not a Python `int` fixed ahead of time), tracing it once covers every possible size. Growing it doesn't trigger a new trace, only the classical gate-counting arithmetic has to scale.

In [ ]:
from qrisp import count_ops, QuantumFloat, measure

trace_count = [0]


@count_ops(meas_behavior="0")
def main(i):
    trace_count[0] += 1
    a = QuantumFloat(i)
    b = QuantumFloat(i)
    c = a + b
    return measure(c)


start = time.perf_counter()
result_small = main(5)
elapsed = (time.perf_counter() - start) * 1000
print(f"main(5): {elapsed:.1f} ms -> {result_small}")

start = time.perf_counter()
result_large = main(1_000_000)
elapsed = (time.perf_counter() - start) * 1000
print(f"main(1_000_000): {elapsed:.1f} ms -> total gates = {sum(result_large.values())}")

print(f"\nNumber of traces: {trace_count[0]}")

main(5): 132.2 ms -> {'t': 8, 'h': 12, 't_dg': 8, 'cx': 50, 's': 4, 'measure': 9}
main(1_000_000): 6.6 ms -> total gates = 22999976

Number of traces: 1


Observations:

- `main(1_000_000)` adds two 1,000,000-qubit `QuantumFloat`s (a circuit with 23 million gates) and counting them still takes a few milliseconds.

- `trace_count` proves why: `i` is a dynamic argument, so growing it from 5 to 1,000,000 doesn't retrace at all! Only the classical gate-counting logic has to scale, not the number of times Python gets involved.

- You'll see exactly the same `trace_count` trick again in the next notebook (`JaspJIT`, in its `Recompilation Diagnosis` section), making the same point for plain JAX code.

If you want to see this at scale, please check out [*Using `BigInteger` to compile Shor's algorithm at 2048 bit*](https://www.qrisp.eu/general/tutorial/BigInteger.html) in the official qrisp documentation. It applies this exact `count_ops` technique to resource-estimate Shor's algorithm at a cryptographically relevant size.

## ... what comes next?

Next up: how `jax.jit` takes this jaxpr and actually compiles it into something fast.